# Libertad Económica y Crecimiento — Análisis de Panel Data

**Autor:** Gabriel López del Cerro | https://www.linkedin.com/in/gabriellopezdelcerro/

Análisis empírico de la relación entre libertad económica y bienestar en **164 países** entre **2000 y 2023**, usando el índice *Economic Freedom of the World* del Fraser Institute y datos del World Bank.

### Pregunta de investigación
¿Existe una relación robusta entre la libertad económica de un país y su nivel de bienestar y crecimiento? ¿Cuál de las 5 dimensiones del índice Fraser tiene mayor impacto?

### Marco teórico

El análisis parte de la premisa de que las instituciones económicas — la calidad del sistema legal, la estabilidad monetaria, la apertura comercial y el grado de intervención regulatoria — son determinantes fundamentales del crecimiento de largo plazo y del bienestar de la población.

Un entorno donde los precios operan libremente, los contratos se respetan y la regulación no distorsiona los incentivos permite que el conocimiento disperso en la sociedad se coordine de manera eficiente, generando prosperidad sostenida. Por el contrario, la interferencia sistemática en estos mecanismos tiende a producir asignaciones ineficientes de recursos, menor inversión y deterioro del nivel de vida.

Este marco sugiere que la riqueza de los países no es un accidente geográfico ni histórico, sino en gran medida el resultado de las reglas del juego que cada sociedad construye y mantiene a lo largo del tiempo.

### Fuentes de datos
- [Fraser Institute — Economic Freedom of the World 2024](https://www.fraserinstitute.org/economic-freedom/dataset)
- [World Bank Open Data API](https://data.worldbank.org)

### Estructura del notebook
1. Instalación de dependencias
2. Carga y limpieza del índice Fraser (ETL)
3. Descarga de datos del World Bank (API)
4. Merge y construcción del panel final
5. Visualización: scatter libertad vs. crecimiento
6. Visualización: mapa coroplético animado
7. Análisis por cuartiles de libertad económica
8. Regresión de panel con fixed effects
9. Exportar visualizaciones

---

## 1. Instalación de dependencias

Instalamos las librerías necesarias. Solo es necesario correr esta celda una vez por sesión.

In [1]:
!pip install pandas requests openpyxl pyarrow plotly linearmodels kaleido -q

## 2. Carga y limpieza del índice Fraser (ETL)

El dataset del Fraser Institute no permite descarga automática. Seguí estos pasos:

1. Descargá el archivo XLSX desde [fraserinstitute.org/economic-freedom/dataset](https://www.fraserinstitute.org/economic-freedom/dataset)
2. Hacé click en el botón **"Download XLSX file"**
3. Corré la celda siguiente y subí el archivo cuando se solicite

El índice cubre **165 jurisdicciones** desde 1970 hasta 2023, con 5 áreas principales:
- **Área 1:** Tamaño del gobierno
- **Área 2:** Sistema legal y derechos de propiedad
- **Área 3:** Moneda sana
- **Área 4:** Libertad de comercio internacional
- **Área 5:** Regulación

In [2]:
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import files

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


In [3]:
uploaded = files.upload()
nombre_archivo = list(uploaded.keys())[0]
print("Archivo cargado:", nombre_archivo)

Saving efotw-2025-master-index-data-for-researchers-iso.xlsx to efotw-2025-master-index-data-for-researchers-iso (5).xlsx
Archivo cargado: efotw-2025-master-index-data-for-researchers-iso (5).xlsx


In [4]:
xl = pd.ExcelFile(nombre_archivo)
print("Hojas disponibles:", xl.sheet_names)

Hojas disponibles: ['EFW Index 1970-2023', 'EFW Panel Dataset', 'EFW Ratings 1950-1965']


In [5]:
df_raw = pd.read_excel(nombre_archivo, sheet_name="EFW Index 1970-2023", header=3)

cols = {
    "Year": "year",
    "ISO_Code": "iso3",
    "Countries": "country",
    "ECONOMIC FREEDOM ALL AREAS": "efw_index",
    "Area 1 Size of Government": "area1_gov_size",
    "Area 2 Legal System & Property Rights -- With Gender Adjustment": "area2_legal",
    "Area 3 Sound Money": "area3_money",
    "Area 4 Freedom to trade internationally": "area4_trade",
    "Area 5 Regulation": "area5_reg",
    "World Bank Region": "region",
    "World Bank Current Income Classification, 1990-Present": "income_group"
}

df = df_raw[list(cols.keys())].rename(columns=cols)
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["efw_index"] = pd.to_numeric(df["efw_index"], errors="coerce")
df = df.dropna(subset=["year", "iso3", "efw_index"])
df = df.sort_values(["iso3", "year"]).reset_index(drop=True)

print("Filas:", len(df))
print("Paises:", df["iso3"].nunique())
print("Anios:", int(df["year"].min()), "-", int(df["year"].max()))
df.head()

Filas: 4558
Paises: 165
Anios: 1970 - 2023


,year,iso3,country,efw_index,area1_gov_size,area2_legal,area3_money,area4_trade,area5_reg,region,income_group
0,2000,AGO,Angola,3.74,5.190192,3.150189,2.441184,NaN,4.196374,Sub-Saharan Africa,Low Income
1,2001,AGO,Angola,3.76,5.252089,3.152605,2.441184,NaN,4.196374,Sub-Saharan Africa,Low Income
2,2002,AGO,Angola,3.85,5.602232,3.179177,2.441184,NaN,4.196374,Sub-Saharan Africa,Low Income
3,2003,AGO,Angola,3.68,5.823192,3.303618,2.441184,NaN,3.149480,Sub-Saharan Africa,Low Income
4,2004,AGO,Angola,3.63,6.964551,3.370286,0.000000,NaN,4.200124,Sub-Saharan Africa,Lower-Middle Income


## 3. Descarga de datos del World Bank (API)

Usamos la API REST del World Bank para obtener 4 indicadores clave para cada país y año:

| Indicador | Código | Descripción |
|-----------|--------|-------------|
| PBI per cápita | NY.GDP.PCAP.KD | USD constantes 2015 |
| Crecimiento PBI per cápita | NY.GDP.PCAP.KD.ZG | Variación % anual |
| Coeficiente de Gini | SI.POV.GINI | Desigualdad de ingresos |
| Mortalidad infantil | SP.DYN.IMRT.IN | Por cada 1.000 nacidos vivos |

In [6]:
def get_wb_indicator(code, name):
    url = f"https://api.worldbank.org/v2/country/all/indicator/{code}"
    params = {"format": "json", "per_page": 20000, "mrv": 54}
    r = requests.get(url, params=params, timeout=30)
    data = r.json()[1]
    rows = []
    for d in data:
        if d["value"] is not None and d["countryiso3code"]:
            rows.append({"iso3": d["countryiso3code"], "year": int(d["date"]), name: d["value"]})
    return pd.DataFrame(rows)

indicadores = {
    "NY.GDP.PCAP.KD":    "gdp_pc",
    "NY.GDP.PCAP.KD.ZG": "gdp_growth",
    "SI.POV.GINI":       "gini",
    "SP.DYN.IMRT.IN":    "infant_mort"
}

print("Descargando datos del World Bank...")
dfs = [get_wb_indicator(code, name) for code, name in indicadores.items()]

df_wb = dfs[0]
for d in dfs[1:]:
    df_wb = df_wb.merge(d, on=["iso3", "year"], how="outer")

print("Filas WB:", len(df_wb))
print("Paises WB:", df_wb["iso3"].nunique())
df_wb.head()

Descargando datos del World Bank...
Filas WB: 13460
Paises WB: 260


,iso3,year,gdp_pc,gdp_growth,gini,infant_mort
0,ABW,1986,16150.651073,NaN,NaN,NaN
1,ABW,1987,18992.068378,17.593206,NaN,NaN
2,ABW,1988,22468.507120,18.304687,NaN,NaN
3,ABW,1989,24730.396448,10.066932,NaN,NaN
4,ABW,1990,24763.653810,0.134480,NaN,NaN


## 4. Merge y construcción del panel final

Combinamos el índice Fraser con los datos del World Bank usando el código ISO3 y el año como llaves de unión. Usamos `inner join` para quedarnos solo con observaciones donde tenemos ambas fuentes disponibles.

In [7]:
df_final = df.merge(df_wb, on=["iso3", "year"], how="inner")
df_final = df_final.dropna(subset=["efw_index", "gdp_growth"])
df_final = df_final.sort_values(["iso3", "year"]).reset_index(drop=True)

print("Panel final:")
print("Filas:", len(df_final))
print("Paises:", df_final["iso3"].nunique())
print("Anios:", int(df_final["year"].min()), "-", int(df_final["year"].max()))
df_final.head(10)

Panel final:
Filas: 4408
Paises: 164
Anios: 1975 - 2023


,year,iso3,country,efw_index,area1_gov_size,area2_legal,area3_money,area4_trade,area5_reg,region,income_group,gdp_pc,gdp_growth,gini,infant_mort
0,2000,AGO,Angola,3.74,5.190192,3.150189,2.441184,NaN,4.196374,Sub-Saharan Africa,Low Income,2172.463267,-0.302928,51.9,115.0
1,2001,AGO,Angola,3.76,5.252089,3.152605,2.441184,NaN,4.196374,Sub-Saharan Africa,Low Income,2189.173522,0.769185,NaN,111.9
2,2002,AGO,Angola,3.85,5.602232,3.179177,2.441184,NaN,4.196374,Sub-Saharan Africa,Low Income,2404.977921,9.857802,NaN,108.1
3,2003,AGO,Angola,3.68,5.823192,3.303618,2.441184,NaN,3.149480,Sub-Saharan Africa,Low Income,2403.404152,-0.065438,NaN,104.0
4,2004,AGO,Angola,3.63,6.964551,3.370286,0.000000,NaN,4.200124,Sub-Saharan Africa,Lower-Middle Income,2583.324115,7.486047,NaN,99.3
5,2005,AGO,Angola,3.80,6.311031,3.373903,0.024335,5.846233,3.425174,Sub-Saharan Africa,Lower-Middle Income,2843.396636,10.067359,NaN,94.4
6,2006,AGO,Angola,4.17,4.787209,3.411700,2.582505,5.801607,4.275297,Sub-Saharan Africa,Lower-Middle Income,3065.055762,7.795575,NaN,89.1
7,2007,AGO,Angola,4.09,4.422909,3.407846,2.660245,5.654178,4.324302,Sub-Saharan Africa,Lower-Middle Income,3336.363325,8.851635,NaN,83.8
8,2008,AGO,Angola,4.18,4.048021,3.408357,3.570460,5.671350,4.225092,Sub-Saharan Africa,Lower-Middle Income,3559.195225,6.678886,42.7,78.7
9,2009,AGO,Angola,4.71,7.066084,3.420744,3.518812,5.597033,3.967416,Sub-Saharan Africa,Lower-Middle Income,3494.807245,-1.809060,NaN,73.7


## 5. Scatter: libertad económica vs. crecimiento

Cada punto representa un país (promedio 2000-2023). El tamaño del punto refleja el PBI per cápita promedio, y el color indica la región geográfica.

La línea punteada muestra la tendencia lineal. Países de interés están etiquetados explícitamente.

In [8]:
df_scatter = df_final.groupby(["iso3", "country", "region", "income_group"]).agg(
    efw_index  = ("efw_index",  "mean"),
    gdp_growth = ("gdp_growth", "mean"),
    gdp_pc     = ("gdp_pc",     "mean")
).reset_index()

df_scatter2 = df_scatter[
    (df_scatter["efw_index"] >= 0) & (df_scatter["efw_index"] <= 10) &
    (df_scatter["gdp_growth"] >= -10) & (df_scatter["gdp_growth"] <= 15)
].copy()

destacados = ["USA", "CHE", "SGP", "HKG", "NZL", "VEN", "ZWE",
              "ARG", "CHL", "URY", "CHN", "SWE", "DNK", "IRL", "IRN"]
df_scatter2["label"] = df_scatter2["iso3"].apply(lambda x: x if x in destacados else "")

fig = px.scatter(
    df_scatter2, x="efw_index", y="gdp_growth", color="region",
    size="gdp_pc", size_max=40, hover_name="country", text="label",
    hover_data={"iso3": False, "gdp_pc": ":,.0f", "efw_index": ":.2f", "gdp_growth": ":.2f"},
    labels={
        "efw_index":  "Indice de Libertad Economica (Fraser, promedio)",
        "gdp_growth": "Crecimiento PBI per capita promedio (%)",
        "region":     "Region",
        "gdp_pc":     "PBI per capita (USD)"
    },
    title="Libertad economica vs. crecimiento — 164 paises (2000-2023)"
)

x = df_scatter2["efw_index"]
y = df_scatter2["gdp_growth"]
z = np.polyfit(x, y, 1)
p = np.poly1d(z)
x_line = np.linspace(x.min(), x.max(), 100)
fig.add_scatter(x=x_line, y=p(x_line), mode="lines", name="Tendencia",
                line=dict(color="black", width=1.5, dash="dash"))

fig.update_traces(textposition="top center", textfont_size=10)
fig.update_layout(
    height=600, template="plotly_white",
    xaxis=dict(range=[3, 10], title="Indice de Libertad Economica (0-10)"),
    yaxis=dict(range=[-8, 15])
)
fig.show()

## 6. Mapa coroplético animado (2000–2023)

El mapa muestra el índice de libertad económica de cada país por año. Usá el botón **Play** para ver la animación, o arrastrá el slider para explorar un año específico.

**Escala de colores:**
- Rojo oscuro → baja libertad económica
- Ámbar → libertad media
- Azul oscuro → alta libertad económica

In [9]:
df_mapa = df_final[["iso3", "country", "year", "efw_index", "region"]].dropna(subset=["efw_index"]).copy()
df_mapa = df_mapa[df_mapa["year"] >= 2000].copy()
df_mapa["year"] = df_mapa["year"].astype(int).astype(str)

fig_mapa2 = px.choropleth(
    df_mapa, locations="iso3", color="efw_index", hover_name="country",
    hover_data={"iso3": False, "efw_index": ":.2f"}, animation_frame="year",
    color_continuous_scale=[(0.0, "#A32D2D"), (0.3, "#EF9F27"), (0.6, "#85B7EB"), (1.0, "#0C447C")],
    range_color=[2, 10], labels={"efw_index": "Libertad Economica"},
    title="Indice de Libertad Economica por pais (Fraser Institute, 2000-2023)"
)

fig_mapa2.update_layout(
    height=550,
    geo=dict(showframe=False, showcoastlines=True, projection_type="natural earth"),
    coloraxis_colorbar=dict(title="Indice (0-10)"),
    sliders=[{"currentvalue": {"prefix": "Ano: "}}]
)

fig_mapa2.layout.sliders[0].steps = sorted(
    fig_mapa2.layout.sliders[0].steps, key=lambda s: int(s["label"])
)
fig_mapa2.show()

## 7. Análisis por cuartiles de libertad económica

Dividimos los 164 países en 4 grupos iguales (cuartiles) según su índice promedio de libertad económica y comparamos tres indicadores de bienestar:

- **PBI per cápita promedio** (nivel de riqueza)
- **Crecimiento del PBI per cápita** (dinamismo económico)
- **Mortalidad infantil** (indicador de bienestar social)

In [10]:
df_cuartil = df_final.groupby(["iso3", "country"]).agg(
    efw_index   = ("efw_index",   "mean"),
    gdp_growth  = ("gdp_growth",  "mean"),
    gdp_pc      = ("gdp_pc",      "mean"),
    infant_mort = ("infant_mort", "mean")
).reset_index()

df_cuartil["cuartil"] = pd.qcut(
    df_cuartil["efw_index"], q=4,
    labels=["Q1 Menos libre", "Q2", "Q3", "Q4 Mas libre"]
)

resumen = df_cuartil.groupby("cuartil", observed=True).agg(
    paises      = ("iso3",        "count"),
    efw_index   = ("efw_index",   "mean"),
    gdp_growth  = ("gdp_growth",  "mean"),
    gdp_pc      = ("gdp_pc",      "mean"),
    infant_mort = ("infant_mort", "mean")
).round(2)

print(resumen.to_string())

                paises  efw_index  gdp_growth    gdp_pc  infant_mort
cuartil                                                             
Q1 Menos libre      41       5.05        1.44   2402.43        53.28
Q2                  41       6.01        2.34   4981.19        39.47
Q3                  41       6.76        2.48  10886.90        18.94
Q4 Mas libre        41       7.72        1.98  31812.69         7.65


In [11]:
cuartiles = ["Q1 Menos libre", "Q2", "Q3", "Q4 Mas libre"]
colores   = ["#A32D2D", "#EF9F27", "#85B7EB", "#0C447C"]

fig_q = make_subplots(rows=1, cols=3, horizontal_spacing=0.1)

fig_q.add_trace(go.Bar(x=cuartiles, y=resumen["gdp_pc"].values, marker_color=colores,
    text=[f"${v:,.0f}" for v in resumen["gdp_pc"].values],
    textposition="outside", showlegend=False), row=1, col=1)

fig_q.add_trace(go.Bar(x=cuartiles, y=resumen["gdp_growth"].values, marker_color=colores,
    text=[f"{v:.2f}%" for v in resumen["gdp_growth"].values],
    textposition="outside", showlegend=False), row=1, col=2)

fig_q.add_trace(go.Bar(x=cuartiles, y=resumen["infant_mort"].values, marker_color=colores,
    text=[f"{v:.1f}" for v in resumen["infant_mort"].values],
    textposition="outside", showlegend=False), row=1, col=3)

fig_q.update_layout(
    title=dict(
        text="Libertad economica y bienestar — comparacion por cuartiles (164 paises, 2000-2023)",
        x=0.5, xanchor="center", y=0.97
    ),
    annotations=[
        dict(text="PBI per capita<br>promedio (USD)",
             x=0.13, y=1.12, xref="paper", yref="paper",
             showarrow=False, font=dict(size=12), align="center"),
        dict(text="Crecimiento PBI<br>per capita (%)",
             x=0.5, y=1.12, xref="paper", yref="paper",
             showarrow=False, font=dict(size=12), align="center"),
        dict(text="Mortalidad infantil<br>(por mil)",
             x=0.87, y=1.12, xref="paper", yref="paper",
             showarrow=False, font=dict(size=12), align="center"),
    ],
    height=580,
    template="plotly_white",
    margin=dict(t=150)
)
fig_q.update_yaxes(rangemode="tozero")
fig_q.show()

## 8. Regresión de panel con fixed effects

Para ir más allá del análisis descriptivo, estimamos modelos de **panel data con fixed effects** de país y año. Esto nos permite:

- **Controlar por características invariantes de cada país** (geografía, historia, cultura)
- **Controlar por shocks globales de cada año** (crisis financieras, pandemias)
- Identificar el efecto *within* — cambios en la libertad económica *dentro* de cada país a lo largo del tiempo

Los errores estándar se clusterean por país para corregir correlación serial.

### Modelos estimados
- **Modelo 1:** EFW (índice global) → crecimiento
- **Modelo 2:** EFW + control de convergencia condicional (log PBI per cápita rezagado)
- **Modelo 3:** Las 5 áreas del EFW por separado → para identificar cuál impulsa más el crecimiento

In [12]:
from linearmodels.panel import PanelOLS

df_panel = df_final[["iso3", "year", "gdp_growth", "efw_index",
                      "area1_gov_size", "area2_legal", "area3_money",
                      "area4_trade", "area5_reg", "gdp_pc"]].copy()

df_panel = df_panel.sort_values(["iso3", "year"])
df_panel["log_gdp_pc"] = np.log(df_panel["gdp_pc"])
df_panel["lag_log_gdp_pc"] = df_panel.groupby("iso3")["log_gdp_pc"].shift(1)
df_panel = df_panel.dropna(subset=["gdp_growth", "efw_index", "lag_log_gdp_pc"])
df_panel = df_panel.set_index(["iso3", "year"])

print("Observaciones:", len(df_panel))
print("Paises:", df_panel.index.get_level_values("iso3").nunique())
df_panel.head()

Observaciones: 4244
Paises: 164


gdp_growth  efw_index  area1_gov_size  area2_legal  area3_money  \
iso3 year                                                                    
AGO  2001    0.769185       3.76        5.252089     3.152605     2.441184   
     2002    9.857802       3.85        5.602232     3.179177     2.441184   
     2003   -0.065438       3.68        5.823192     3.303618     2.441184   
     2004    7.486047       3.63        6.964551     3.370286     0.000000   
     2005   10.067359       3.80        6.311031     3.373903     0.024335   

           area4_trade  area5_reg       gdp_pc  log_gdp_pc  lag_log_gdp_pc  
iso3 year                                                                   
AGO  2001          NaN   4.196374  2189.173522    7.691279        7.683617  
     2002          NaN   4.196374  2404.977921    7.785296        7.691279  
     2003          NaN   3.149480  2403.404152    7.784641        7.785296  
     2004          NaN   4.200124  2583.324115    7.856832        7.784641  
     2005     5.846233   3.425174  2843.396636    7.952755        7.856832

In [13]:
mod1 = PanelOLS(dependent=df_panel["gdp_growth"], exog=df_panel[["efw_index"]],
                entity_effects=True, time_effects=True)
res1 = mod1.fit(cov_type="clustered", cluster_entity=True)

mod2 = PanelOLS(dependent=df_panel["gdp_growth"], exog=df_panel[["efw_index", "lag_log_gdp_pc"]],
                entity_effects=True, time_effects=True)
res2 = mod2.fit(cov_type="clustered", cluster_entity=True)

df_areas = df_panel.dropna(subset=["area1_gov_size", "area2_legal", "area3_money", "area4_trade", "area5_reg"])
mod3 = PanelOLS(dependent=df_areas["gdp_growth"],
                exog=df_areas[["area1_gov_size", "area2_legal", "area3_money",
                               "area4_trade", "area5_reg", "lag_log_gdp_pc"]],
                entity_effects=True, time_effects=True)
res3 = mod3.fit(cov_type="clustered", cluster_entity=True)

print("=" * 55)
print("MODELO 1 - EFW solo (FE pais + año)")
print("=" * 55)
print(res1.summary.tables[1])

print("\n" + "=" * 55)
print("MODELO 2 - EFW + convergencia (FE pais + año)")
print("=" * 55)
print(res2.summary.tables[1])

print("\n" + "=" * 55)
print("MODELO 3 - Las 5 areas por separado")
print("=" * 55)
print(res3.summary.tables[1])

MODELO 1 - EFW solo (FE pais + año)
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
efw_index      0.7723     0.2724     2.8351     0.0046      0.2382      1.3064

MODELO 2 - EFW + convergencia (FE pais + año)
                               Parameter Estimates                                
                Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
----------------------------------------------------------------------------------
efw_index          1.0659     0.2948     3.6151     0.0003      0.4879      1.6440
lag_log_gdp_pc    -3.7859     0.8772    -4.3156     0.0000     -5.5057     -2.0660

MODELO 3 - Las 5 areas por separado
                               Parameter Estimates                                
                Parameter  Std. Err.     T-stat    P-value    Lowe

In [14]:
areas  = ["Tamano gobierno", "Sistema legal", "Moneda sana", "Comercio libre", "Regulacion"]
coefs  = [-0.2497,  0.0344,  0.2098,  0.3513,  0.7359]
lower  = [-0.6887, -0.6362, -0.0097,  0.0384,  0.2249]
upper  = [ 0.1893,  0.7050,  0.4294,  0.6641,  1.2470]
pvals  = [ 0.2648,  0.9198,  0.0610,  0.0278,  0.0048]

colores     = ["#A32D2D" if p > 0.05 else "#0C447C" for p in pvals]
error_minus = [c - l for c, l in zip(coefs, lower)]
error_plus  = [u - c for c, u in zip(coefs, upper)]

fig_coef = go.Figure()
fig_coef.add_trace(go.Bar(
    x=areas, y=coefs, marker_color=colores,
    error_y=dict(type="data", symmetric=False, array=error_plus, arrayminus=error_minus,
                 color="gray", thickness=1.5, width=6),
    text=[f"p={p}" for p in pvals], textposition="outside"
))
fig_coef.add_hline(y=0, line_dash="dash", line_color="black", line_width=1)
fig_coef.update_layout(
    title="Efecto de cada area de libertad economica sobre el crecimiento del PBI<br><sup>Panel FE (pais + año), errores clustered por pais. Azul = significativo (p menor a 0.05)</sup>",
    yaxis_title="Coeficiente (puntos de crecimiento por punto de indice)",
    xaxis_title="Area del indice Fraser Institute",
    template="plotly_white", height=500, showlegend=False
)
fig_coef.show()

### Interpretación de resultados

| Área | Coeficiente | p-value | Conclusión |
|------|-------------|---------|------------|
| Regulación (área 5) | +0.74 | 0.005 | Significativa |
| Comercio libre (área 4) | +0.35 | 0.028 | Significativa |
| Moneda sana (área 3) | +0.21 | 0.061 | Marginal |
| Sistema legal (área 2) | +0.03 | 0.920 | No significativa |
| Tamaño del gobierno (área 1) | -0.25 | 0.265 | No significativa |

#### ¿Qué nos dicen estos números?

El resultado más importante no es el coeficiente más alto — es el patrón que emerge cuando se ven los cinco juntos.

Las dos dimensiones que impactan significativamente el crecimiento son la **desregulación** y la **libertad comercial**. Ambas tienen algo en común: son el grado en que el Estado se abstiene de distorsionar las decisiones descentralizadas de millones de agentes económicos. Cuando las empresas pueden contratar, producir y comerciar sin fricciones artificiales, los recursos fluyen hacia sus usos más productivos sin necesidad de coordinación central.

El **tamaño del gobierno** no resulta significativo — y esto es un matiz importante. El problema no es cuánto gasta el Estado, sino cómo interfiere. Un Estado grande que provee bienes públicos eficientemente es menos dañino que uno pequeño que distorsiona precios, restringe la competencia o impone regulaciones arbitrarias. El dato empírico sugiere que la narrativa de "menos Estado = más crecimiento" es una simplificación: lo que importa es la *calidad* de la intervención, no su tamaño.

El **sistema legal** tampoco muestra efecto significativo dentro del período estudiado. Una posible explicación: la calidad institucional y legal opera en horizontes de tiempo muy largos — sus efectos se acumulan en décadas, no en años — y los fixed effects de país que usa este modelo ya capturan gran parte de esa variación histórica entre países. Esto no implica que las instituciones no importen; implica que su impacto es estructural y se refleja en el nivel de riqueza alcanzado, no en las variaciones anuales del crecimiento.

La **moneda sana** aparece marginalmente significativa (p=0.061). Los episodios de alta inflación destruyen el sistema de precios como mecanismo de información: cuando los precios nominales se mueven por razones monetarias y no por razones reales, los agentes no pueden distinguir señales de ruido, la inversión se paraliza y el capital se fuga hacia activos de reserva de valor. El efecto existe pero es difícil de identificar con precisión en un panel tan heterogéneo, donde conviven países con inflación crónica y países con estabilidad monetaria histórica.

#### La lectura de fondo

Los resultados apuntan a una conclusión que va más allá de la econometría: el crecimiento no lo generan los gobiernos mediante la asignación de recursos, sino los individuos y las empresas cuando operan en un entorno de reglas predecibles, precios libres y comercio abierto. La intervención regulatoria y las barreras comerciales no son políticas neutras con costos distribuidos uniformemente — son impuestos implícitos sobre la actividad productiva, cuyos efectos negativos este análisis cuantifica en casi un punto porcentual de crecimiento anual por cada punto de deterioro en el índice.

En perspectiva: un punto porcentual de crecimiento adicional sostenido durante 30 años duplica el ingreso per cápita. La diferencia entre una economía libre y una intervenida no se siente en el corto plazo — se acumula silenciosamente hasta producir las brechas de bienestar que el análisis por cuartiles documenta.

## 9. Exportar visualizaciones

Guardamos las 4 visualizaciones como PNG de alta resolución para usar en el README de GitHub y en el post de LinkedIn.

In [15]:
!pip install -U kaleido -q
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [16]:
!pip install kaleido==0.2.1 -q

In [17]:
fig.write_image("scatter_libertad_crecimiento.png", scale=2)
fig_mapa2.write_image("mapa_coropletico.png", scale=2)
fig_q.write_image("cuartiles_bienestar.png", scale=2)
fig_coef.write_image("regresion_coeficientes.png", scale=2)
print("Imagenes guardadas correctamente.")

Imagenes guardadas correctamente.


In [1]:
from google.colab import files

files.download("scatter_libertad_crecimiento.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
files.download("mapa_coropletico.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
files.download("cuartiles_bienestar.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
files.download("regresion_coeficientes.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>